# 13 · Skills：发现、选择与读取各有边界

Skill 目录让模型知道有哪些方法，正文才给执行方法。若把所有正文立即塞进上下文，会浪费预算，也扩大不相关指令的影响。
但延迟读取引入新问题：同名包怎么选，引用的资源属于谁，读取途中策略变化怎么办？
本章在新目录中创建两个合成 Skill；它们是输入数据，不会安装到个人 Skill 目录，也不执行包中的指令。

**运行范围**：本章直接使用最终包中的共享组件，逐格验证本专题机制；不会启动模型、访问真实收费 API 或使用个人凭据。每次从第一格顺序运行，运行文件进入独立的 `runs/` 目录。改动参数后，从相应状态初始化格重新运行。


In [1]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'code/mini_codex/__init__.py').is_file())
sys.path.insert(0, str(ROOT/'code'))
from mini_codex.learning import load_example, show_source, fresh_run_dir, run_python, run_chapter
from unittest import TestCase
expect = TestCase()  # 只借用异常断言；下面逐格调用实现，不运行总测试替代实验。


## 源码事实与教学选择

固定对照提交：`c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee`。以下链接说明 Codex 对应机制；Python 类型和实验状态机是教学选择，运行它们不等于运行 Codex。

- [S13-02 · parse_skill_file](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/ext/skills/src/loader/host.rs#L347-L394)：Host 发现阶段读取完整 SKILL.md 以解析 frontmatter，同时加载可选元数据；所以按需进入模型上下文不等于磁盘从未读过正文。
- [S13-14 · SkillsThreadState::read_skill](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/ext/skills/src/state.rs#L409-L468)：cloud 读取即使有缓存也先检查当前开关与 auth generation，读后再次检查 generation；失效则拒绝。
- [S13-16 · page_response](https://github.com/openai/codex/blob/c0cc9fc820f62e4dbdde68214dd6d792d6c8a7ee/codex-rs/ext/skills/src/tools/read.rs#L473-L510)：分页计算完整序列化响应字节数，按 UTF-8 字符边界搜索可容纳正文；预算无法容纳内容时报错。


## 实验 1 · 目录只提供候选，精确来源解决同名


教学 loader 只解析单行 name/description frontmatter；固定 Codex 的 Host 发现会读取完整文件，
所以不能把“正文尚未注入模型”解释成“磁盘从未读正文”。两个 authority 提供同名方法时，名称查询应明确失败。


In [2]:
skills = load_example('13', 'skill_catalog')
run_dir = fresh_run_dir('ch13')
roots = {name: run_dir/name for name in ('host','team')}
for name, root in roots.items():
    package = root/'task-review'
    package.mkdir(parents=True)
    (package/'SKILL.md').write_text('---\nname: Review Tasks\ndescription: Check task-list contracts\n---\nRead examples.md before editing.\n')
    (package/'examples.md').write_text('检查 status 字段；先验证再提交。\n' * 40)
allowed = {'host','team'}
loader = skills.Skills(roots, lambda authority: authority in allowed)
catalog = loader.discover('step-1')
assert len(catalog.entries) == 2
with expect.assertRaisesRegex(skills.SkillError, 'selection_ambiguous'):
    loader.select(catalog, name='Review Tasks')
entry = loader.select(catalog, locator='skill://team/task-review')
assert entry.authority == 'team'
print(catalog.display())


({'name': 'Review Tasks', 'description': 'Check task-list contracts', 'package': 'skill://host/task-review'}, {'name': 'Review Tasks', 'description': 'Check task-list contracts', 'package': 'skill://team/task-review'})


## 实验 2 · 正文与资源一起提交


选择后再读正文和明确引用的资源，返回内容摘要以便追溯。一个资源失败时，不给调用者半份 bundle。
这不回滚已经发生的磁盘读取，而是限制哪些片段可发布给后续请求。


In [3]:
bundle = loader.load(catalog, entry, ('examples.md',))
assert [f.resource for f in bundle.fragments] == ['SKILL.md','examples.md']
assert len(bundle.fragments[1].sha256) == 64
before = sum(e['event'] == 'bundle_committed' for e in loader.events)
with expect.assertRaisesRegex(skills.SkillError, 'resource_missing'):
    loader.load(catalog, entry, ('missing.md',))
after = sum(e['event'] == 'bundle_committed' for e in loader.events)
assert before == after
assert loader.events[-1]['event'] == 'bundle_failed'
print({'loaded': [f.resource for f in bundle.fragments], 'failed_load_published': False})


{'loaded': ['SKILL.md', 'examples.md'], 'failed_load_published': False}


## 实验 3 · 资源路径不能偷换 authority


路径检查拒绝父目录穿越与 symlink；伪造 Entry 根同样失败。这里是应用级防误用，
并不是能抵御恶意并发替换文件的 OS sandbox。把相对资源名改为合法的 examples.md，对照正常读取。


In [4]:
from dataclasses import replace
with expect.assertRaisesRegex(skills.SkillError, 'invalid_resource'):
    loader.load(catalog, entry, ('../secret.md',))
link = entry.root/'linked.md'
link.symlink_to(entry.root/'examples.md')
with expect.assertRaisesRegex(skills.SkillError, 'symlink_rejected'):
    loader.load(catalog, entry, ('linked.md',))
forged = replace(entry, root=roots['host']/'task-review')
with expect.assertRaisesRegex(skills.SkillError, 'source_mismatch'):
    loader.read_page(forged, 'examples.md', 256)
assert loader.load(catalog, entry, ('examples.md',)).fragments[1].text
print('路径、链接和 authority/root 绑定均拒绝了替换。')


路径、链接和 authority/root 绑定均拒绝了替换。


## 实验 4 · 分页预算包括元数据，游标绑定内容


正文含中文，因此不能按任意字节截断 UTF-8。分页按完整序列化响应计预算，并把内容摘要放进游标身份。
把 `PAGE_BYTES` 改为 180 或 400，页数会变化，拼回内容应不变；太小的预算必须显式失败。


In [5]:
PAGE_BYTES = 256
pages, cursor = [], None
while True:
    page = loader.read_page(entry, 'examples.md', PAGE_BYTES, cursor=cursor)
    assert len(page.wire().encode()) <= PAGE_BYTES
    pages.append(page)
    cursor = page.next_cursor
    if cursor is None:
        break
assert ''.join(p.contents for p in pages) == (entry.root/'examples.md').read_text()
with expect.assertRaisesRegex(skills.SkillError, 'response_budget'):
    loader.read_page(entry, 'examples.md', 10)
old_cursor = pages[0].next_cursor
assert old_cursor is not None
(entry.root/'examples.md').write_text('更新后的内容\n' * 40)
with expect.assertRaisesRegex(skills.SkillError, 'invalid_cursor'):
    loader.read_page(entry, 'examples.md', PAGE_BYTES, cursor=old_cursor)
print({'page_count': len(pages), 'max_wire_bytes': max(len(p.wire().encode()) for p in pages)})


{'page_count': 13, 'max_wire_bytes': 256}


## 实验 5 · 旧目录不能保留已撤销的读取权


目录是某一步的快照，读取权则要实时复查。取消也只能丢弃暂存 bundle，不能让半成品进入请求。
恢复时重新允许来源并重新读取，不能把“曾经选中过”当永久许可。


In [6]:
allowed.remove('team')
with expect.assertRaisesRegex(skills.SkillError, 'source_denied'):
    loader.load(catalog, entry)
allowed.add('team')
with expect.assertRaisesRegex(skills.SkillError, 'cancelled'):
    loader.load(catalog, entry, ('examples.md',), cancelled=lambda: True)
recovered = loader.load(catalog, entry, ('examples.md',))
assert '更新后的内容' in recovered.fragments[1].text
assert loader.events[-1]['event'] == 'bundle_committed'
print('旧目录被实时权限拦截；取消未发布 bundle；恢复读到新内容。')


旧目录被实时权限拦截；取消未发布 bundle；恢复读到新内容。


## 你现在可以解释什么

能指出目录快照、资源内容身份和实时权限三个不同的寿命。教学实现没有 cloud provider、完整 YAML 解析器、安装器、模型上下文投影或 OS 隔离。

**验证边界**：本章断言验证当前局部实现与合成输入。未运行固定提交的 Rust 测试，未验证真实模型、外部服务或累计 Agent 的端到端行为。
